## Example 1: Tool execution pattern (conceptual demo)

In [ ]:
from typing import List, Dict, Any
from dataclasses import dataclass

# Example 1: Tool execution pattern (conceptual demo)
@dataclass
class Tool:
    """Simple tool representation"""
    name: str
    description: str
    func: callable
    
    def execute(self, **kwargs) -> str:
        """Execute the tool with given arguments"""
        return self.func(**kwargs)

# Create sample tools
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression"""
    try:
        return str(eval(expression))
    except:
        return "Invalid expression"

def string_reverser(text: str) -> str:
    """Reverse a string"""
    return text[::-1]

# Register tools
tools = [
    Tool("calculator", "Math evaluator", calculator),
    Tool("string_reverser", "Reverses text", string_reverser)
]

print("Tool execution system:")
for tool in tools:
    print(f"  - {tool.name}: {tool.description}")
print(f"\nExample: calculator('2+2') = {tools[0].execute(expression='2+2')}")
print(f"Example: string_reverser('hello') = {tools[1].execute(text='hello')}")

## Example 2: Validation pattern for safe tool execution

In [ ]:
from typing import Dict, Any, Optional

# Example 2: Validation pattern for safe tool execution
class ValidationNode:
    """Node that validates inputs before tool execution"""
    
    def __init__(self, tools: list, fallback_on_error: bool = True):
        self.tools = {t.name: t for t in tools}
        self.fallback_on_error = fallback_on_error
        
    def validate_and_execute(self, tool_name: str, args: Dict[str, Any]) -> str:
        """Validate inputs and execute tool safely"""
        if tool_name not in self.tools:
            return f"Error: Unknown tool {tool_name}"
        
        if self.fallback_on_error:
            try:
                return self.tools[tool_name](**args)
            except Exception as e:
                return f"Validation failed: {str(e)}"
        return self.tools[tool_name](**args)

# Safe division tool
def safe_divide(a: float, b: float) -> float:
    """Safely divide two numbers"""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

# Create validation node
val_node = ValidationNode([], fallback_on_error=True)
print("ValidationNode created with fallback enabled")
print("Features: Input validation, error handling, safe execution")
print("Example: Prevents division by zero and returns error message")

## Example 3: ReAct agent pattern (Reason + Act)

In [ ]:
from typing import List, Dict, Any
from collections import deque

# Example 3: ReAct agent pattern (Reason + Act)
class ReActAgent:
    """Simple ReAct agent implementation"""
    
    def __init__(self, tools: List, recursion_limit: int = 10):
        self.tools = {t['name']: t['func'] for t in tools}
        self.recursion_limit = recursion_limit
        self.history = deque(maxlen=recursion_limit)
        
    def reason(self, query: str) -> str:
        """Reasoning step - decide which tool to use"""
        if "weather" in query.lower():
            return "get_weather"
        elif "search" in query.lower():
            return "search_web"
        return "unknown"
    
    def act(self, tool_name: str, args: dict) -> str:
        """Action step - execute the chosen tool"""
        if tool_name in self.tools:
            return self.tools[tool_name](**args)
        return f"Tool {tool_name} not found"

# Mock tools
tools = [
    {"name": "get_weather", "func": lambda loc: f"Weather in {loc}: Sunny, 22°C"},
    {"name": "search_web", "func": lambda q: f"Results for '{q}': Found 10 articles"}
]

agent = ReActAgent(tools, recursion_limit=10)
print("ReAct Agent created with:")
print(f"  Tools: {list(agent.tools.keys())}")
print(f"  Recursion limit: {agent.recursion_limit}")
print("  Pattern: Reason → Act → Observe → Loop")

## Example 4: State injection pattern for context-aware tools

In [ ]:
from typing import Dict, Any
from functools import wraps

# Example 4: State injection pattern for context-aware tools
class StateInjector:
    """Inject state and store into tool functions"""
    
    def __init__(self):
        self.state = {}
        self.store = {}
    
    def inject_state(self, func):
        """Decorator to inject state into tool function"""
        @wraps(func)
        def wrapper(*args, **kwargs):
            kwargs['state'] = self.state
            return func(*args, **kwargs)
        return wrapper
    
    def inject_store(self, func):
        """Decorator to inject store for persistence"""
        @wraps(func)
        def wrapper(*args, **kwargs):
            kwargs['store'] = self.store
            return func(*args, **kwargs)
        return wrapper

injector = StateInjector()
injector.state = {"user_id": "user123", "session": "abc"}
injector.store = {"cache": {}}

@injector.inject_state
def get_user_info(state: dict) -> str:
    return f"Current user: {state.get('user_id', 'unknown')}"

print("State Injection Pattern:")
print(f"  State: {injector.state}")
print(f"  Store: {injector.store}")
print(f"  Result: {get_user_info()}")

## Example 5: Conditional workflow routing pattern

In [ ]:
from typing import Dict, List, Optional
from enum import Enum

# Example 5: Conditional workflow routing pattern
class ToolCondition(Enum):
    """Conditions for tool routing"""
    CONTINUE = "continue"
    END = "end"
    ERROR = "error"

class WorkflowRouter:
    """Route workflow based on tool execution results"""
    
    def __init__(self, max_iterations: int = 5):
        self.max_iterations = max_iterations
        self.current_iteration = 0
        
    def tools_condition(self, result: Dict) -> ToolCondition:
        """Determine next step based on tool result"""
        self.current_iteration += 1
        
        if "error" in result:
            return ToolCondition.ERROR
        elif self.current_iteration >= self.max_iterations:
            return ToolCondition.END
        elif result.get("complete", False):
            return ToolCondition.END
        else:
            return ToolCondition.CONTINUE
    
    def reset(self):
        """Reset iteration counter"""
        self.current_iteration = 0

router = WorkflowRouter(max_iterations=5)
results = [
    {"status": "processing"},
    {"status": "processing"},
    {"complete": True}
]

print("Workflow Router Configuration:")
print(f"  Max iterations: {router.max_iterations}")
print("\nRouting simulation:")
for i, res in enumerate(results, 1):
    condition = router.tools_condition(res)
    print(f"  Step {i}: {res} → {condition.value}")